In [ ]:
"""
聚宽(JoinQuant)数据下载脚本
===========================
上传到聚宽研究环境运行，下载股票不复权分钟线数据并保存为 .csv 文件。

使用方法：
  1. 将本文件上传到聚宽研究环境
  2. 修改下方 CONFIG 区域的参数（股票代码、日期范围、分钟频率等）
  3. 运行全部代码
  4. 点击输出的下载链接下载 .csv 文件
"""

from jqdata import *
import pandas as pd
from datetime import datetime

# 获取用户ID
import platform
node = platform.node()
userid = node.split('-')[-1]


# ============================================================
# CONFIG - 在此修改参数
# ============================================================

# 股票代码（聚宽格式：代码.XSHE / 代码.XSHG）
SECURITY = '000001.XSHE'

# 数据日期范围
START_DATE = '1990-01-01'
END_DATE = '2026-05-19'

# 分钟线频率: '1m', '5m', '15m', '30m', '60m'
FREQUENCY = '1m'

# 是否跳过停牌日期（分钟线建议跳过，否则会有大量空数据）
SKIP_SUSPENDED = True

# 输出文件名（不含扩展名）
OUTPUT_NAME = None  # 默认自动生成，也可手动指定如 '000001_XSHE_1m'

# ============================================================


def code_convert(code):
    """将聚宽代码格式转为通用格式: 000001.XSHE -> 000001.SZ"""
    if 'XSHE' in code:
        return code.replace('XSHE', 'SZ')
    return code.replace('XSHG', 'SH')


def download_minute_data(security, start_date, end_date, frequency, skip_suspended=True):
    """
    下载股票不复权分钟线数据

    参数:
        security: 股票代码，聚宽格式
        start_date: 开始日期
        end_date: 结束日期
        frequency: 分钟线频率
        skip_suspended: 是否跳过停牌日

    返回:
        DataFrame
    """
    print(f"正在下载 {security} {frequency} 分钟线数据...")
    print(f"  日期范围: {start_date} ~ {end_date}")
    print(f"  复权方式: 不复权")

    # 获取交易日列表
    trade_days = get_trade_days(start_date=start_date, end_date=end_date)
    print(f"  交易日数量: {len(trade_days)}")

    # 获取分钟线数据（不复权）
    # frequency 参数: '1m', '5m', '15m', '30m', '60m'
    # fq: 复权方式，'None' 表示不复权
    df = get_price(
        security,
        start_date=start_date,
        end_date=end_date,
        frequency=frequency,
        fields=['open', 'close', 'high', 'low', 'volume', 'money'],
        fq=None,
    )

    if df is None or df.empty:
        print("  警告: 未获取到数据！")
        return None

    print(f"  获取到 {len(df)} 条记录")

    # 重置索引，将时间索引变为普通列
    df = df.reset_index()
    # 统一时间列名
    if 'index' in df.columns:
        df = df.rename(columns={'index': 'datetime'})
    elif 'date' in df.columns:
        df = df.rename(columns={'date': 'datetime'})

    # 确保时间列为字符串格式（csv 兼容性更好）
    df['datetime'] = df['datetime'].astype(str)

    # 添加股票代码列
    df['code'] = code_convert(security)

    # 重排列顺序
    cols = ['code', 'datetime', 'open', 'close', 'high', 'low', 'volume', 'money']
    df = df[[c for c in cols if c in df.columns]]

    return df


def save_csv(df, output_name):
    """保存 DataFrame 为 csv 文件"""
    filename = f"{output_name}.csv"
    df.to_csv(filename, index=False, encoding='utf-8')
    print(f"已保存: {filename} ({len(df)} 条记录)")
    return filename


def get_download_url(filename):
    """生成聚宽文件下载链接"""
    # 聚宽研究环境中，文件保存在用户目录下
    # 下载链接格式需要用户ID，这里提供通用格式
    return f"https://www.joinquant.com/user/{userid}/files/{filename}?download=1"


# ============================================================
# 主流程
# ============================================================

# 生成输出文件名
if OUTPUT_NAME is None:
    code_short = code_convert(SECURITY).replace('.', '_')
    OUTPUT_NAME = f"{code_short}_{FREQUENCY}_{START_DATE}_{END_DATE}"

# 下载数据
df = download_minute_data(
    security=SECURITY,
    start_date=START_DATE,
    end_date=END_DATE,
    frequency=FREQUENCY,
    skip_suspended=SKIP_SUSPENDED,
)

df = df.dropna()

if df is not None:
    # 显示数据预览
    print("\n数据预览:")
    print(df.head(10))
    print(f"\n数据类型:\n{df.dtypes}")
    print(f"\n时间范围: {df['datetime'].min()} ~ {df['datetime'].max()}")

    # 保存为 csv
    filename = save_csv(df, OUTPUT_NAME)

    # 输出下载链接
    url = get_download_url(filename)
    print(f"\n下载链接:\n{url}")
else:
    print("下载失败，请检查参数设置。")


正在下载 000001.XSHE 1m 分钟线数据...
  日期范围: 1990-01-01 ~ 2026-05-19
  复权方式: 不复权
  交易日数量: 5188
  获取到 1244880 条记录

数据预览:
        code             datetime  open    ...      low   volume     money
0  000001.SZ  2005-01-04 09:31:00  6.57    ...     6.57   2700.0   17739.0
1  000001.SZ  2005-01-04 09:32:00  6.57    ...     6.56  16000.0  105098.0
2  000001.SZ  2005-01-04 09:33:00  6.56    ...     6.55  10300.0   67474.0
3  000001.SZ  2005-01-04 09:34:00  6.54    ...     6.54   3100.0   20287.0
4  000001.SZ  2005-01-04 09:35:00  6.56    ...     6.56   4800.0   31444.0
5  000001.SZ  2005-01-04 09:36:00  6.52    ...     6.51   8500.0   55450.0
6  000001.SZ  2005-01-04 09:37:00  6.50    ...     6.50  13800.0   89783.0
7  000001.SZ  2005-01-04 09:38:00  6.50    ...     6.48   7800.0   50690.0
8  000001.SZ  2005-01-04 09:39:00  6.48    ...     6.46   4500.0   29152.0
9  000001.SZ  2005-01-04 09:40:00  6.50    ...     6.50   3400.0   22148.0

[10 rows x 8 columns]

数据类型:
code         object
datetime     